In [8]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

# Step 1 - load everything
candidates_df = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df = pd.read_parquet(Path("../../top20_df.parquet"))
source_chunks = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

# Step 2 - filter candidates to top 20 sourcbte docs only
top_source_ids = set(top20_df["source_doc_id"].tolist())

candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

print(f"Filtered candidates: {len(candidates_filtered)}")
print(candidates_filtered.columns.tolist())

Filtered candidates: 0
['suspicious_chunk_id', 'suspicious_doc_id', 'suspicious_chunk_index', 'suspicious_start_char', 'suspicious_end_char', 'source_chunk_id', 'source_doc_id', 'source_chunk_index', 'source_start_char', 'source_end_char', 'embedding_score', 'embedding_rank']


In [11]:
# Determine the text column name in each parquet (preprocessing may differ)
susp_text_col = "embedding_text"
src_text_col  = "chunk_text"

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", susp_text_col]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", susp_text_col: "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", src_text_col]]
    .rename(columns={"chunk_id": "source_chunk_id", src_text_col: "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)

# Keep top- highest-similarity pairs per source doc to feed the LLM
TOP_PAIRS_PER_DOC = 10
top_pairs = (
    pairs_df
    .sort_values("embedding_score", ascending=False)
    .groupby("source_doc_id")
    .head(TOP_PAIRS_PER_DOC)
    .reset_index(drop=True)
)

print(f"Source docs to evaluate: {top_pairs['source_doc_id'].nunique()}")
print(f"Total pairs sent to LLM: {len(top_pairs)}")
top_pairs[["source_doc_id", "embedding_score", "suspicious_text", "source_text"]].head()


Source docs to evaluate: 0
Total pairs sent to LLM: 0


,source_doc_id,embedding_score,suspicious_text,source_text


In [ ]:
llm_results = []
from tqdm import tqdm
groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="Pair Creation"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")
    break
pairs

0it [00:00, ?it/s]


[{'suspicious_text': 'and no easy one. "Happy I must contrive that they shall be," she thought, "for unhappiness and discontent are among the foxes that spoil the vines. Stupid they shall not be, while I can think of any force to stir their brains; they have ordinary intelligence, all of them, and they shall learn to use it; dull and sleepy children I can\'t abide. Fairly good they will be, if they are busy and happy, and clever enough to see the folly of being anything but good! And so, month after month, for many years to come, I must be helping Nancy and Kathleen to be the right sort of women, and wives, and mothers, and Gilbert and Peter the proper kind of men, and husbands, and fathers. Mother Carey\'s chickens must be able to show the good birds the way home, as the Admiral said, and I should think they ought to be able to set a few bad birds on the right track now and then!" Well, all this would be a task to frighten and stagger many a person, but it only kindled Mrs. Carey\'s l

In [ ]:
import time, json, re
from ollama import chat
from tqdm import tqdm

OLLAMA_MODEL = "gemma4:e4b"

def score_source_doc(source_doc_id: str, pairs: list[dict]) -> dict:
    pairs_text = "\n\n".join([
        f"[Pair {i+1}]\n"
        f"SUSPICIOUS: {p['suspicious_text'][:600]}\n"
        f"SOURCE CANDIDATE: {p['source_text'][:600]}"
        for i, p in enumerate(pairs)
    ])

    prompt = (
        f"You are a plagiarism detection expert.\n"
        f"Below are {len(pairs)} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Analyze whether the suspicious chunks appear to be copied, paraphrased, or otherwise "
        f"derived from the source document. "
        f"Score the overall likelihood that this source document is the true origin of the "
        f"suspicious text (0.0 = definitely not, 1.0 = definitely yes). "
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

    t0 = time.time()
    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False
    )
    elapsed = time.time() - t0

    full_response = response.message.content
    match = re.search(r'\{.*\}', full_response, re.DOTALL)
    if not match:
        print(f"[DEBUG] Raw response:\n{full_response[:500]}")
        raise ValueError("No JSON found in model response")

    # Fix invalid escape sequences the model sometimes emits (e.g. \s, \p, \d)
    raw_json = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', match.group())
    data = json.loads(raw_json)

    return {
        "source_doc_id":        source_doc_id,
        "llm_score":            float(data.get("score", 0.0)),
        "llm_is_likely_source": bool(data.get("is_likely_source", False)),
        "llm_reasoning":        data.get("reasoning", ""),
        "elapsed_s":            round(elapsed, 1),
    }


llm_results = []

groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")

    try:
        llm_results.append(score_source_doc(source_doc_id, pairs))
    except Exception as e:
        print(f"[WARN] Error scoring {source_doc_id}: {e}")
        llm_results.append({
            "source_doc_id": source_doc_id,
            "llm_score": 0.0,
            "llm_is_likely_source": False,
            "llm_reasoning": f"Error: {e}",
            "elapsed_s": 0.0,
        })

llm_scores_df = (
    pd.DataFrame(llm_results)
    .sort_values("llm_score", ascending=False)
    .reset_index(drop=True)
)
llm_scores_df


<!-- Save LLM score df -->

Save Most likely source documents according to llm validaiton of spans

In [18]:
llm_scores_df.to_parquet("llm_scores_df.parquet",index=False)